In [8]:
using MyPackage
using MyPackage.Geometry
using MyPackage.VLM

using Printf

function new_plane()
    plain = Airfoil("../assets/airfoils/Plain/Plain.dat")
    wing = Surface(
        airfoils=[plain, plain],
        b=6.0,
        chord=y -> 2 * (1 - y^2)^0.5,
        sw_center=0.5,
    )
    return Plane([wing])
end

function new_plane2()
    s1223 = Airfoil("../assets/airfoils/S1223/S1223.dat")
    wing = Surface(
        airfoils=[s1223, s1223],
        b=0.850,
        chord=y -> 0.2,
        sw_center=0.5,
        pos=(0.6, 0.0, 0.0)
    )
    hstab = Surface(
        airfoils=[s1223, s1223],
        b=0.34,
        chord=y -> 0.1,

    )
    return Plane([wing, hstab])
end

function run_example()
    println("\n" * "="^72)
    println("VLM Solver Example")
    println("="^72)

    V_mag = 10.0
    alpha_deg = 5.0
    beta_deg = 0.0
    rho = 1.225
    mu = 1.81 * 10^(-5)
    S_ref = 0.85*0.2
    # S_ref = 3 * pi
    q_inf = 0.5 * rho * V_mag^2

    plane = new_plane2()

    println("\nFreestream: V = $(V_mag) m/s, alpha = $(alpha_deg)°, beta = $(beta_deg)°")
    println("Reference area: $(S_ref) m²")
    println("Q inf: $(q_inf)")
    println("CG: $(plane.data.CG)")

    t0 = time()
    FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz = VLMSolver(
        plane,
        V_mag,
        (alpha_deg, beta_deg);
        n_chordxspan=[(20, 60), (20, 30)],
        rho=rho,
        epsilon2=10^(-10)
    )
    elapsed = time() - t0

    CL = L[1] / (q_inf * S_ref)
    CD = D[1] / (q_inf * S_ref)
    CL_trefftz = L_trefftz[1] / (q_inf * S_ref)
    CD_trefftz = D_trefftz[1] / (q_inf * S_ref)

    Re = rho * V_mag * plane.surfaces[1].MAC / mu

    println("\nResults")
    println("-"^72)
    @printf("Solve time       : %.3f s\n", elapsed)
    @printf("Reynolds number  : %.1f\n", Re)
    @printf("FX               : %.6f N\n", FX[1])
    @printf("FY               : %.6f N\n", FY[1])
    @printf("FZ               : %.6f N\n", FZ[1])
    @printf("Lift             : %.6f N\n", L[1])
    @printf("Drag             : %.6f N\n", D[1])
    @printf("CL               : %.6f\n", CL)
    @printf("CD               : %.6f\n", CD)
    @printf("Trefftz lift     : %.6f N\n", L_trefftz[1])
    @printf("Trefftz drag     : %.6f N\n", D_trefftz[1])
    @printf("Trefftz CL       : %.6f N\n", CL_trefftz)
    @printf("Trefftz CD       : %.6f N\n", CD_trefftz)

    return FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz
end

result = run_example()


VLM Solver Example
Loading airfoil: S1223 from ../assets/airfoils/S1223/S1223.dat

Freestream: V = 10.0 m/s, alpha = 5.0°, beta = 0.0°
Reference area: 0.17 m²
Q inf: 61.25000000000001
CG: (0.65, 0.0, 0.0)

Results
------------------------------------------------------------------------
Solve time       : 3.913 s
Reynolds number  : 135359.1
FX               : 0.352284 N
FY               : 0.000000 N
FZ               : 10.269784 N
Lift             : 10.200001 N
Drag             : 1.246014 N
CL               : 0.979592
CD               : 0.119665
Trefftz lift     : 11.949269 N
Trefftz drag     : 1.239561 N
Trefftz CL       : 1.147589 N
Trefftz CD       : 0.119045 N


([0.35228412899335515, 0.07249135513399285], [[-0.09516575250887087, -0.010226820530062214, -0.004246859524000115, -0.0018266956048206306, -0.000463169949830544, 0.0005312041578112679, 0.0013625715913452249, 0.0020984821251448737, 0.002760202952219239, 0.0033525072749551587  …  0.003368100770209853, 0.002778835560393391, 0.0021205229206497363, 0.0013882873996901215, 0.0005604860340386788, -0.00043181491301782306, -0.001800966285287085, -0.004278817536606558, -0.011433538898263203, -0.15701502224024122], [-0.008709954829043671, -0.0007109790454227413, 0.00014751879881697935, 0.0007701618955254207, 0.0013218082895155984, 0.0018078690346808827, 0.0022140297826648866, 0.0025301645156243498, 0.0027538469775908437, 0.0028895413376778113  …  0.002894069326780926, 0.0027594724040427556, 0.0025371348913236255, 0.002222654593965505, 0.0018185274487044559, 0.00133489835550914, 0.0007857284091562266, 0.00016237210690207891, -0.0007628205673470363, -0.014975740685734523]], [4.773486012530289e-16, -